# Correlation matrix plotting
Visualises functional connectivity matrices at the participant, group, and dataset level. Execute cells top to bottom. Only cells marked **[CONFIGURE]** require changes.

> Run `scripts/run_scripts.ipynb` before running this notebook.

## 1. Environment Setup **[OPTIONAL]**
Mounts Google Drive and installs dependencies when running on Colab. Skip if running locally.

> **NOTE: requires moving `PROJECT` folder to Google Drive.**

In [ ]:
# COLAB
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd '/content/drive/My Drive/PROJECT'

    !pip install -q numpy pandas scipy scikit-learn matplotlib seaborn

    sys.path.insert(0, './product/src')

## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.spatial.distance import jensenshannon
from preprocessing.timeseries import preprocess_timeseries
from data_io.save_load_dataset import load_dataset
from utils.paths import get_project_root

root = get_project_root()

## 3. Configuration **[CONFIGURE]**

| Variable | Description |
|---|---|
| `PARTICIPANT_IDX` | Row index into metadata for single-participant plots |
| `RANDOM_STATE` | Random seed for reproducible group sampling |
| `GROUP_SAMPLE_N` | Number of participants per group in the ASD vs TD comparison |

In [ ]:
PARTICIPANT_IDX = 704   # row index into metadata for single-participant plots
RANDOM_STATE    = 42    # for reproducible group sampling
GROUP_SAMPLE_N  = 5     # participants per group in the ASD vs TD comparison

# Diverging colour map
custom_cmap = mcolors.LinearSegmentedColormap.from_list(
    "SalmonSlate", ["cornflowerblue", "#FFFFFF", "salmon"]
)

## 4. Load Dataset

Loads the full ABIDE dataset and converts Fisher z-scores to Pearson r for plotting. Update the prefix argument in `load_dataset` to use a subset.

In [ ]:
prefix = '1100'
X, X_raw, y, metadata, feature_labels = load_dataset(prefix, verbose=True)

# Convert Fisher z to Pearson r for plotting
X_pearson     = np.tanh(X)
X_pearson_raw = np.tanh(X_raw)

## 5. Reconstruct Connectivity Matrix

Loads the raw timeseries for the selected participant, reconstructs their symmetric 200x200 connectivity matrix from the upper triangle of Pearson r values, and sets the diagonal to 1.

In [ ]:
idx = metadata['X'].index[PARTICIPANT_IDX]
pid = metadata.loc[idx, 'ID']

path        = root / f"product/data/raw/{prefix}/{pid}_rois_cc200.1D"
ts          = np.genfromtxt(path)
participant = preprocess_timeseries(ts)

print(f"Participant: {pid} (idx={idx})")
print(metadata.iloc[idx])

N_ROIS = 200
matrix = np.zeros((N_ROIS, N_ROIS))
iu     = np.triu_indices(N_ROIS, k=1)
matrix[iu] = X_pearson[idx]
matrix     = matrix + matrix.T
np.fill_diagonal(matrix, 1)

## 6. Full Connectivity Matrix Heatmap

Plots the full 200x200 connectivity matrix for the selected participant. Uncomment the `savefig` line to save as PNG.

In [ ]:
plt.figure(figsize=(10, 8), dpi=300)

ax = sns.heatmap(matrix,
                 cmap=custom_cmap,
                 vmin=-1, vmax=1,
                 center=0,
                 square=True,
                 cbar=True,
                 xticklabels=False,
                 yticklabels=False)

sns.despine(left=True, bottom=True, top=True, right=True)
# plt.savefig("connectivity_matrix.png", transparent=True, bbox_inches='tight')
plt.show()

## 7. Annotated Heatmap (First 5 ROI Pairs)

Plots an annotated heatmap of the first 5x5 submatrix, with abbreviated ROI labels.

In [ ]:
small_view = matrix[:5, :5]
abbrev_i   = feature_labels[:5].abbrev_i
abbrev_j   = feature_labels[:5].abbrev_j

pd.set_option('display.max_colwidth', None)

plt.figure(figsize=(8, 7), dpi=300)

ax = sns.heatmap(small_view,
                 vmin=-1, vmax=1,
                 center=0,
                 cmap=custom_cmap,
                 annot=True,
                 fmt=".2f",
                 xticklabels=abbrev_i,
                 yticklabels=abbrev_j,
                 square=True,
                 linewidths=1,
                 linecolor='black',
                 annot_kws={"size": 10})

sns.despine(left=True, bottom=True, top=True, right=True)
plt.xticks(rotation=90, ha='right')
plt.yticks(rotation=0)
plt.show()

## 8. Upper Triangle as a Feature Vector

Extracts the upper triangle of the 5x5 submatrix and plots it as a single-row heatmap. Illustrates how the symmetric matrix is flattened into the feature vector stored in X.

In [ ]:
iu          = np.triu_indices_from(small_view, k=1)
vector_view = small_view[iu].reshape(1, -1)

plt.figure(figsize=(12, 2), dpi=300)

sns.heatmap(vector_view,
            vmin=-1, vmax=1,
            center=0,
            cmap=custom_cmap,
            annot=True,
            fmt=".2f",
            xticklabels=False,
            yticklabels=False,
            cbar_kws={"orientation": "horizontal", "pad": 0.4},
            linewidths=1,
            linecolor='black')

plt.show()

## 9. Default Mode Network Submatrix

Plots the connectivity matrix for the first 10 Default Mode Network (DMN) ROIs. ROI IDs are taken from `feature_labels` where either endpoint belongs to the DMN.

In [ ]:
# Collect all ROI IDs that appear in either column of the DMN
dmn_rois_i  = feature_labels.loc[feature_labels['network_i'] == 'DMN', 'roi_i']
dmn_rois_j  = feature_labels.loc[feature_labels['network_j'] == 'DMN', 'roi_j']
dmn_roi_ids = sorted(pd.concat([dmn_rois_i, dmn_rois_j]).unique())

print(f"DMN ROIs: {dmn_roi_ids}")

# Build ROI-id -> abbreviation label mapping
labels_map = pd.concat([
    feature_labels[['roi_i', 'abbrev_i']].rename(columns={'roi_i': 'id', 'abbrev_i': 'label'}),
    feature_labels[['roi_j', 'abbrev_j']].rename(columns={'roi_j': 'id', 'abbrev_j': 'label'}),
]).drop_duplicates('id').set_index('id')

dmn_matrix     = matrix[np.ix_(dmn_roi_ids, dmn_roi_ids)]
reduced        = dmn_matrix[:10, :10]
labels_reduced = [labels_map.loc[i, 'label'] for i in dmn_roi_ids[:10]]

plt.figure(figsize=(10, 8), dpi=300)

ax = sns.heatmap(reduced,
                 annot=True,
                 vmin=-1, vmax=1,
                 fmt=".2f",
                 cmap=custom_cmap,
                 center=0,
                 xticklabels=labels_reduced,
                 yticklabels=labels_reduced,
                 linewidths=1,
                 linecolor='black',
                 square=True)

plt.title("Representative Connectivity Matrix (DMN ROIs 1-10)")
cbar = ax.collections[0].colorbar
cbar.outline.set_edgecolor('black')
cbar.outline.set_linewidth(1.3)
plt.show()

## 10. Group-Level Connectivity Profile

Plots all participants sorted by diagnostic label (TD then ASD), with one row per participant and one column per connectivity feature. Gives a broad sense of group-level FC differences across the dataset.

In [ ]:
idx_sorted = np.argsort(y)
X_plot     = X_pearson[idx_sorted]

plt.figure(figsize=(10, 6), dpi=300)
sns.heatmap(X_plot, cmap=custom_cmap, center=0, vmin=-1, vmax=1, xticklabels=False)
plt.xlabel("Connectivity Features")
plt.ylabel("Participants")
plt.title("Group-Level Connectivity Profile")
plt.show()

## 11. ASD vs TD Comparison

Samples `GROUP_SAMPLE_N` participants from each diagnostic group, identifies the 10 features with the largest absolute mean difference between groups, and plots them as a heatmap. A horizontal line separates the TD and ASD rows.

In [ ]:
# Sample and sort by diagnostic group
orig_indices    = metadata.groupby('DX_GROUP').sample(n=GROUP_SAMPLE_N, random_state=RANDOM_STATE)
orig_indices    = orig_indices.sort_values('DX_GROUP')
balanced_sample = orig_indices.reset_index(drop=True)

X_10 = X_pearson[orig_indices.index]
p_10 = balanced_sample['ID']
y_10 = balanced_sample['DX_GROUP']

# Compute per-group means and rank features by absolute difference
group_0 = X_10[y_10.values == 0]
group_1 = X_10[y_10.values == 1]
delta   = group_1.mean(axis=0) - group_0.mean(axis=0)

feature_ranking = pd.DataFrame({
    'Feature':        feature_labels['desc_feature_name'],
    'Feature_abbrev': feature_labels['abbrev_feature_name'],
    'Difference':     delta,
    'Abs_Difference': np.abs(delta),
})
top_features = feature_ranking.sort_values('Abs_Difference', ascending=False).head(10)
print("Top 10 most discriminative features:")
print(top_features[['Feature', 'Difference']])

X_filtered   = X_10[:, top_features.index]
f_top_abbrev = top_features['Feature_abbrev'].values

# Plot with a horizontal line separating the two groups
group_map   = {0: 'TD', 1: 'ASD'}
y_labels    = y_10.map(group_map)
yticklabels = [f"ID: {i} ({l})" for i, l in zip(p_10, y_labels)]

plt.figure(figsize=(10, 8), dpi=300)

ax = sns.heatmap(X_filtered,
                 cmap=custom_cmap,
                 center=0,
                 vmin=-1, vmax=1,
                 xticklabels=f_top_abbrev,
                 yticklabels=yticklabels,
                 cbar_kws={'label': 'Correlation Coefficient'})

plt.axhline(y=GROUP_SAMPLE_N, color='black', linewidth=3)
plt.title("Connectivity Features: ASD vs TD", fontsize=14, pad=20)
plt.xlabel("ROI-to-ROI Features", fontsize=12, labelpad=20)
plt.ylabel("Participants", fontsize=12, labelpad=20)
plt.show()

## 12. Feature Matrix Schematic

Plots a schematic of the full feature matrix X, sampling a small number of rows and columns to illustrate its shape.

In [ ]:
n_participants, n_features = X_pearson.shape

rand_participants = sorted(np.random.choice(range(4, n_participants - 2), 4, replace=False))
rand_features     = sorted(np.random.choice(range(4, n_features - 2), 4, replace=False))

row_idx = [0, 1, 2, 3] + list(rand_participants) + [n_participants - 2, n_participants - 1]
col_idx = [0, 1, 2, 3] + list(rand_features)     + [n_features - 2,     n_features - 1]

real_slice = X[np.ix_(row_idx, col_idx)]

participant_labels     = [str(i + 1) if i not in rand_participants else "..." for i in row_idx]
feature_labels_display = [str(i + 1) if i not in rand_features   else "..." for i in col_idx]

limit = np.max(np.abs(X))

plt.figure(figsize=(12, 8), dpi=300)

ax = sns.heatmap(real_slice,
                 annot=True,
                 fmt=".2f",
                 vmin=-limit, vmax=limit,
                 center=0,
                 cmap=custom_cmap,
                 xticklabels=feature_labels_display,
                 yticklabels=participant_labels,
                 linewidths=1,
                 linecolor='black',
                 square=True,
                 annot_kws={"size": 9})

cbar = ax.collections[0].colorbar
cbar.outline.set_edgecolor('black')
cbar.outline.set_linewidth(1.3)

plt.xlabel(f"Connectivity Features $N={19900}$", labelpad=10)
plt.ylabel(f"Participants $P={1100}$", labelpad=10)
plt.title("Schematic of feature matrix $X$", pad=15)
plt.show()